In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("TotalPipelineSpendsReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
# =======================================================
# Workload-type mapping (plan §3.1 / §5.5)
# =======================================================
# Friendly label per billing_origin_product. Unknown / new products fall back
# to the raw value downstream (coalesce) so nothing is ever dropped.
WORKLOAD_MAP = {
    "DLT": "DLT Pipeline",
    "SQL": "DBSQL Materialized View",
    "DATABASE": "Online Table",
    "VECTOR_SEARCH": "Vector Search",
    "MODEL_SERVING": "Model Serving",
    "AI_FUNCTIONS": "AI Functions",
}

# Single source of truth for "which workloads are expected to carry a
# system.lakeflow.pipelines snapshot" (plan §5.5). Vector Search / Model
# Serving / AI Functions never get a row, so they are excluded by design.
# Defined here as the canonical set; the backend §5.3 metadata-missing KPI
# mirrors it so the two never drift.
METADATA_BEARING_WORKLOADS = {
    "DLT Pipeline",
    "DBSQL Materialized View",
    "Online Table",
}

In [ ]:
from pyspark.sql.window import Window


# =======================================================
# Total Pipeline Spends Client
# =======================================================
# Sibling of TotalPoolSpendsClient. Rolls the per-cluster
# dbspend360_pipeline_dbu_cost staging table up into the denormalized
# dbspend360_total_pipeline_spends rollup. Differences vs the pool rollup:
#   * source DBU table: dbspend360_pipeline_dbu_cost
#                       (keyed (workspace_id, pipeline_id, usage_date,
#                        cluster_id, billing_origin_product))
#   * target table:     dbspend360_total_pipeline_spends
#                       (keyed (workspace_id, pipeline_id, usage_date,
#                        billing_origin_product) - cluster_id is aggregated
#                        away into compute_mode; billing_origin_product STAYS
#                        in the grain so the per-workload $ split is exact and
#                        reconciles row-for-row with staging - NO within-day
#                        dominant-product approximation. See plan §3.3 / §5.5).
#   * derived dimensions (the rollup is the ONLY place these are derived):
#       - workload_type : friendly label from billing_origin_product via
#                         WORKLOAD_MAP, raw value for unknowns (never dropped).
#       - compute_mode  : serverless / classic / mixed. A single
#                         (pipeline, day, product) can straddle serverless +
#                         classic clusters -> 'mixed'.
#       - cost_basis    : full (serverless) / dbu_only (classic) /
#                         partial (mixed). Drives the per-row UI honesty icon.
#   * metadata denorm:  SCD-collapse system.lakeflow.pipelines on
#                       (workspace_id, pipeline_id) and denormalize
#                       pipeline_name / pipeline_type / created_by / run_as /
#                       delete_time -> pipeline_deleted_at. created_by/run_as
#                       come straight from the system table (99.94% populated
#                       for DLT, plan §0/§3.4) - NO REST API, NO metadata cache
#                       (the key simplification vs the Instance Pools rollup).
#   * metadata_missing: computed BEFORE the COALESCE fallback on pipeline_name
#                       so it captures the underlying snapshot state, not the
#                       post-fallback state. Three-state, product-aware badge
#                       (plan §3.5): active / deleted-visible / metadata-not-
#                       available (the EXPECTED state for Vector Search etc.).
#   * cloud-cost join (v2): classic clusters carry a ClusterId tag on AWS, so
#     their EC2/EBS is already in dbspend360_cloud_cost_explorer keyed by
#     cluster_id (staging keeps cluster_id -> NO re-ingest). Each cluster-day
#     cloud is attributed to its pipeline (DBU-weighted, so a cluster shared
#     across pipelines reconciles instead of double-counting), then spread
#     across the pipeline's product rows by classic-DBU share (plan §3.2).
#     Serverless rows (no separate VM line) keep cloud_cost = NULL -> the UI
#     renders "-" via compute_mode (plan §5), never a misleading $0. The
#     intermediate attribution is reconciled to the explorer per
#     (cluster_id, usage_date, currency) within $0.01 (plan §3.3).
#     total_cost = databricks_cost + COALESCE(cloud_cost, 0) (unchanged).
#   * MERGE key: (workspace_id, pipeline_id, usage_date,
#                 billing_origin_product) - all NON-nullable, so plain '=' is
#                 safe here. The null-safe concern is staging-only (§5.4): the
#                 nullable cluster_id is not in the rollup key.
class TotalPipelineSpendsClient:

    TABLE_NAME = "dbspend360_total_pipeline_spends"

    def __init__(
        self,
        audit_table: str,
        cloud_cost_table: str,
        databricks_cost_table: str,
        target_table: str,
        error_log_table: str,
        overlap_days: int,
        logger=None,
    ):
        self.audit_table = audit_table
        self.cloud_cost_table = cloud_cost_table
        self.databricks_cost_table = databricks_cost_table
        self.target_table = target_table
        self.error_log_table = error_log_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("TotalPipelineSpendsClient")

    def _load_pipeline_snapshot(self):
        # SCD-collapse system.lakeflow.pipelines to one row per
        # (workspace_id, pipeline_id) carrying the most-recent snapshot.
        # QUALIFY ROW_NUMBER() OVER (... ORDER BY change_time DESC) = 1 is
        # holistically safe on tied change_time (one winner per partition).
        # delete_time is non-null iff the pipeline was deleted; carry it
        # through as pipeline_deleted_at for the §3.5 "Deleted YYYY-MM-DD"
        # badge. workspace_id is in the partition because pipeline_id is only
        # unique within a workspace (plan §3.3).
        # The join keys are aliased to p_* so they never collide with the
        # day-grain columns. On serverless Spark Connect a shared-name column
        # that participates in an equi-join key becomes unresolvable when
        # referenced (qualified) after the join, so we keep every post-join
        # column uniquely named and reference only bare names downstream.
        return spark.sql("""
            SELECT workspace_id AS p_workspace_id,
                   pipeline_id  AS p_pipeline_id,
                   name AS pipeline_name,
                   pipeline_type,
                   created_by,
                   run_as,
                   delete_time AS pipeline_deleted_at
            FROM system.lakeflow.pipelines
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY workspace_id, pipeline_id
                ORDER BY change_time DESC) = 1
        """)

    def _assert_reconciliation(self, attributed, cloud_df, start_dt, end_dt):
        # Plan §3.3 invariant: for every (cluster_id, usage_date, currency)
        # present in BOTH the classic staging set and the explorer, the cloud
        # attributed to pipelines must equal explorer.cloud_cost within
        # 0.01 USD. Because the rollup collapses cluster_id away, this is
        # asserted on the INTERMEDIATE attribution, not on the final table.
        # Unmatched explorer clusters (job / all-purpose) are EXPECTED and not
        # checked here; the inner join only sees clusters that exist in both.
        # DBU-proportional shares sum to 1 per cluster-day, so this normally
        # passes by construction - it is a hard guard against a logic bug
        # (e.g. an accidental fan-out that double-counts a cluster's cloud).
        # The one exception is a classic cluster with an explorer EC2 row but
        # $0 staging DBU for the (cluster, day, currency): the cloud has no DBU
        # weight to ride on, so it is logged as a non-fatal
        # CLOUD_UNATTRIBUTED_ZERO_DBU event and skipped (the run continues)
        # rather than failing the whole rollup. Only keys WITH DBU that still
        # diverge are fatal.
        # Carry cluster_dbu_total (a window-constant per cluster-day-currency)
        # into the per-key aggregate so we can tell an availability trap apart
        # from a real attribution bug below.
        attr_sum = (
            attributed
            .groupBy("cluster_id", "usage_date", "currency")
            .agg(
                F.sum(F.coalesce(F.col("attr_cloud_cost"), F.lit(0.0))).alias("attr_cloud_cost"),
                F.max(F.coalesce(F.col("cluster_dbu_total"), F.lit(0.0))).alias("cluster_dbu_total"),
            )
        )
        mismatch_df = (
            attr_sum
            .join(cloud_df, on=["cluster_id", "usage_date", "currency"], how="inner")
            .withColumn("diff", F.abs(F.col("attr_cloud_cost") - F.col("cloud_cost")))
            .filter(F.col("diff") > 0.01)
        )

        # Split mismatches into two classes (correctness-vs-availability):
        #   * unattributable - a classic cluster has an explorer EC2 row but
        #     $0 staging DBU for that (cluster, day, currency), so
        #     cluster_dbu_total is 0 and attr_cloud_cost is NULL->0 BY
        #     CONSTRUCTION (there is no DBU weight for the cloud to ride on).
        #     This is rare (classic clusters normally bill DBU) but it is an
        #     availability trap, NOT an attribution-logic bug: aborting the
        #     whole rollup would block the table over a single EC2-but-$0-DBU
        #     cluster-day. Log it as a non-fatal data-quality event and let the
        #     cloud fall through as an "unknown/-" pipeline-day; the run
        #     continues.
        #   * fatal - cluster_dbu_total > 0 yet the attributed total still
        #     diverges. DBU-proportional shares sum to 1 per cluster-day, so
        #     this can only mean a real fan-out / double-count bug -> hard fail.
        unattributable_df = mismatch_df.filter(F.col("cluster_dbu_total") <= 0)
        fatal_df = mismatch_df.filter(F.col("cluster_dbu_total") > 0)

        def _to_err_rows(df, error_type, detail_prefix):
            return (
                df.select(
                    F.lit("RECONCILIATION").alias("source_system"),
                    F.lit(error_type).alias("error_type"),
                    F.col("cluster_id"),
                    F.lit(None).cast("string").alias("job_id"),
                    F.lit(None).cast("string").alias("run_id"),
                    F.col("usage_date"),
                    F.col("currency"),
                    F.concat(
                        F.lit(detail_prefix),
                        F.col("attr_cloud_cost").cast("string"),
                        F.lit(", explorer="),
                        F.col("cloud_cost").cast("string"),
                        F.lit(", diff="),
                        F.col("diff").cast("string"),
                    ).alias("error_detail"),
                    F.to_json(F.struct(
                        F.col("cluster_id"), F.col("usage_date"), F.col("currency"),
                        F.col("attr_cloud_cost").alias("attributed_cloud_cost"),
                        F.col("cloud_cost").alias("explorer_cloud_cost"),
                        F.col("cluster_dbu_total"),
                        F.col("diff"),
                    )).alias("raw_record"),
                )
                .withColumn("created_at", F.lit(datetime.now(timezone.utc)))
            )

        # Non-fatal: log unattributable EC2 (explorer cost, $0 classic DBU) and
        # keep going so the run does not fail on a rare zero-DBU cluster-day.
        unattributable_count = unattributable_df.count()
        if unattributable_count > 0:
            try:
                _safe_append(
                    _to_err_rows(
                        unattributable_df,
                        "CLOUD_UNATTRIBUTED_ZERO_DBU",
                        "pipeline cloud_cost unattributable ($0 classic DBU): attributed=",
                    ),
                    self.error_log_table,
                )
            except Exception as e:
                self.logger.exception(
                    f"Failed to write unattributable-cloud rows to error log: {e}"
                )
            self.logger.warning(
                "CLOUD_UNATTRIBUTED_ZERO_DBU: %d (cluster_id, usage_date, currency) "
                "keys have explorer EC2 cost but $0 classic DBU; the cloud cannot be "
                "attributed to a pipeline and is left as an unknown/- pipeline-day "
                "(logged to %s, run continues).",
                unattributable_count, self.error_log_table,
            )

        fatal_count = fatal_df.count()
        if fatal_count == 0:
            self.logger.info(
                f"Reconciliation OK: attributed pipeline cloud_cost matches "
                f"{self.cloud_cost_table} ± 0.01 USD for {start_dt} → {end_dt} "
                f"(ignoring {unattributable_count} unattributable zero-DBU keys)."
            )
            return

        try:
            _safe_append(
                _to_err_rows(
                    fatal_df,
                    "CLOUD_COST_MISMATCH",
                    "pipeline cloud_cost mismatch: attributed=",
                ),
                self.error_log_table,
            )
        except Exception as e:
            self.logger.exception(
                f"Failed to write reconciliation mismatches to error log: {e}"
            )

        sample = fatal_df.limit(5).collect()
        sample_str = "; ".join(
            f"(cluster={r['cluster_id']}, date={r['usage_date']}, "
            f"attributed={r['attr_cloud_cost']}, explorer={r['cloud_cost']}, "
            f"diff={r['diff']:.4f})"
            for r in sample
        )
        raise DataQualityError(
            f"Reconciliation invariant violated: {fatal_count} "
            f"(cluster_id, usage_date, currency) keys with non-zero classic DBU "
            f"differ from {self.cloud_cost_table} by > 0.01 USD. Sample: {sample_str}"
        )

    def build_total_pipeline_spends(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(
                f"Building dbspend360_total_pipeline_spends for {start_dt} → {end_dt}"
            )

            staging_df = (
                spark.table(self.databricks_cost_table)
                    .alias("stg")
                    .filter(
                        (F.col("usage_date") >= F.lit(start_dt)) &
                        (F.col("usage_date") <= F.lit(end_dt))
                    )
            )

            if staging_df.limit(1).count() == 0:
                self.logger.info(
                    "No pipeline DBU rows in this date window; nothing to roll up."
                )
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                    "SUCCESS", 0, "No DBU data in window",
                )
                return

            # 1) Collapse staging (per cluster) to pipeline-day-PRODUCT.
            #    billing_origin_product STAYS in the grain (§3.3) so the
            #    per-workload $ split is exact - NO dominant-product collapse.
            #    A single (pipeline, day, product) can still straddle
            #    serverless + classic clusters -> 'mixed' compute_mode.
            day_df = (
                staging_df.groupBy(
                    "workspace_id", "pipeline_id", "usage_date", "billing_origin_product"
                )
                .agg(
                    F.sum("databricks_cost").alias("databricks_cost"),
                    F.sum("update_cost").alias("update_cost"),
                    F.sum("maintenance_cost").alias("maintenance_cost"),
                    F.when(F.countDistinct("compute_mode") > 1, F.lit("mixed"))
                     .otherwise(F.first("compute_mode")).alias("compute_mode"),
                    F.concat_ws(" + ", F.array_sort(F.collect_set("sku_name"))).alias("sku_name"),
                    F.first("currency").alias("currency"),
                )
                .withColumn(
                    "cost_basis",
                    F.when(F.col("compute_mode") == "serverless", F.lit("full"))
                     .when(F.col("compute_mode") == "classic", F.lit("dbu_only"))
                     .otherwise(F.lit("partial")),
                )
            )

            # workload_type from billing_origin_product; unknowns fall back to
            # the raw value (coalesce) so new/unmapped products are never lost.
            mapping = F.create_map([F.lit(x) for kv in WORKLOAD_MAP.items() for x in kv])
            day_df = day_df.withColumn(
                "workload_type",
                F.coalesce(mapping[F.col("billing_origin_product")], F.col("billing_origin_product")),
            )

            # 1b) Cloud-cost attribution (plan §3.2 / §3.3).
            #     Classic DLT clusters carry a ClusterId tag on AWS, so their
            #     EC2/EBS is already in dbspend360_cloud_cost_explorer keyed by
            #     cluster_id. Serverless pipelines (cluster_id IS NULL) have NO
            #     separate VM line and never enter classic_staging, so their
            #     product rows fall through the LEFT join below with
            #     cloud_cost = NULL (the UI renders "-" via compute_mode, §5).
            #     "Serverless" is taken from staging's compute_mode (NOT from
            #     cluster_id alone): serverless Model Serving / Vector Search /
            #     AI Functions endpoints carry a NON-null "-v2n" cluster_id but
            #     run in Databricks' account, so their id exists under no AWS tag
            #     and must be excluded here - otherwise the cloud attribution
            #     would chase a cluster_id that is never in the explorer.
            #     cluster_id IS NOT NULL is retained because the explorer join
            #     is cluster_id-keyed.
            classic_staging = staging_df.filter(
                (F.col("cluster_id").isNotNull())
                & (F.col("compute_mode") == "classic")
            )

            cloud_df = (
                spark.table(self.cloud_cost_table)
                    .filter(
                        (F.col("cost_incurred_date") >= F.lit(start_dt)) &
                        (F.col("cost_incurred_date") <= F.lit(end_dt)) &
                        (F.col("cluster_id").isNotNull())
                    )
                    .select(
                        F.col("cluster_id"),
                        F.col("cost_incurred_date").alias("usage_date"),
                        F.col("currency"),
                        F.col("cloud_cost"),
                    )
                    # Explorer is already at (cluster_id, day, currency) grain;
                    # dropDuplicates is a cheap defensive guard against any
                    # accidental dupes that would fan the join out.
                    .dropDuplicates(["cluster_id", "usage_date", "currency"])
            )

            # 1:1 sanity check (plan §3.2 step 3). DLT classic clusters are
            # pipeline-scoped; if a (cluster_id, usage_date) ever maps to >1
            # pipeline the DBU-weighted attribution below still reconciles
            # (no double-count) - we only log so the anomaly stays visible.
            collisions = (
                classic_staging
                .select("workspace_id", "pipeline_id", "usage_date", "cluster_id")
                .distinct()
                .groupBy("cluster_id", "usage_date")
                .agg(F.countDistinct(
                    F.concat_ws("|", F.col("workspace_id"), F.col("pipeline_id"))
                ).alias("n_pipelines"))
                .filter(F.col("n_pipelines") > 1)
            )
            if collisions.limit(1).count() > 0:
                self.logger.warning(
                    "CLUSTER_PIPELINE_FANOUT: %d (cluster_id, usage_date) pairs "
                    "map to >1 pipeline; cloud cost is split DBU-proportionally "
                    "across them (no double-count, plan §3.2).",
                    collisions.count(),
                )

            # Step 1: attribute each (cluster_id, day, currency) cloud to its
            # pipeline(s), weighted by that pipeline's classic DBU on the
            # cluster. Unmatched clusters (no explorer row yet) yield NULL and
            # are summed away -> the affected pipeline-day stays "unknown".
            cluster_pipe = (
                classic_staging
                .groupBy("workspace_id", "pipeline_id", "usage_date", "cluster_id", "currency")
                .agg(F.sum("databricks_cost").alias("pipe_cluster_dbu"))
                .withColumn(
                    "cluster_dbu_total",
                    F.sum("pipe_cluster_dbu").over(
                        Window.partitionBy("cluster_id", "usage_date", "currency")
                    ),
                )
            )
            attributed = (
                cluster_pipe
                .join(cloud_df, on=["cluster_id", "usage_date", "currency"], how="left")
                .withColumn(
                    "attr_cloud_cost",
                    F.when(
                        F.col("cluster_dbu_total") > 0,
                        F.col("cloud_cost") * F.col("pipe_cluster_dbu") / F.col("cluster_dbu_total"),
                    ),
                )
            )
            attributed = safe_cache(attributed)

            # §3.3 hard gate on the intermediate attribution (the rollup drops
            # cluster_id, so the invariant cannot be asserted on the target).
            self._assert_reconciliation(attributed, cloud_df, start_dt, end_dt)

            pipeline_cloud = (
                attributed
                .groupBy("workspace_id", "pipeline_id", "usage_date")
                .agg(F.sum("attr_cloud_cost").alias("pipeline_cloud_cost"))
            )

            # Step 2: spread each pipeline-day cloud across that pipeline's
            # product rows by classic-DBU share (informed apportionment; the
            # pipeline-day SUM stays exact). Pure-serverless product rows never
            # appear here, so the LEFT join onto day_df leaves them NULL.
            classic_prod = (
                classic_staging
                .groupBy("workspace_id", "pipeline_id", "usage_date", "billing_origin_product")
                .agg(F.sum("databricks_cost").alias("classic_prod_dbu"))
                .withColumn(
                    "pipe_day_classic_dbu",
                    F.sum("classic_prod_dbu").over(
                        Window.partitionBy("workspace_id", "pipeline_id", "usage_date")
                    ),
                )
            )
            cloud_per_product = (
                classic_prod
                .join(pipeline_cloud, on=["workspace_id", "pipeline_id", "usage_date"], how="left")
                .withColumn(
                    "cloud_cost",
                    F.when(
                        F.col("pipe_day_classic_dbu") > 0,
                        F.col("pipeline_cloud_cost") * F.col("classic_prod_dbu") / F.col("pipe_day_classic_dbu"),
                    ),
                )
                .select(
                    "workspace_id", "pipeline_id", "usage_date",
                    "billing_origin_product", "cloud_cost",
                )
            )

            # Attach cloud onto the per-product rows. LEFT so every day_df row
            # (incl. serverless / mixed) survives; serverless -> cloud_cost NULL.
            day_df = day_df.join(
                cloud_per_product,
                on=["workspace_id", "pipeline_id", "usage_date", "billing_origin_product"],
                how="left",
            )

            # 2) SCD-collapse system.lakeflow.pipelines and LEFT-join metadata.
            #    LEFT so pipeline-days with no snapshot row still land in the
            #    rollup; metadata_missing is the signal (set BEFORE the COALESCE
            #    fallback paints a synthetic pipeline_name).
            pipelines_df = self._load_pipeline_snapshot()

            # Join keys live only on the day-grain side (p_* on the snapshot
            # side), so every post-join column is a unique bare name.
            joined = (
                day_df
                .join(
                    pipelines_df,
                    on=(
                        (F.col("workspace_id") == F.col("p_workspace_id")) &
                        (F.col("pipeline_id") == F.col("p_pipeline_id"))
                    ),
                    how="left",
                )
                .withColumn("metadata_missing", F.col("pipeline_name").isNull())
            )

            select_cols = [
                F.col("workspace_id"),
                F.col("pipeline_id"),
                F.col("usage_date"),
                F.coalesce(
                    F.col("pipeline_name"),
                    F.concat(F.lit("Pipeline "), F.col("pipeline_id")),
                ).alias("pipeline_name"),
                F.col("pipeline_type"),
                F.col("created_by"),
                F.col("run_as"),
                F.col("workload_type"),
                F.col("compute_mode"),
                F.col("cost_basis"),
                F.col("metadata_missing"),
                F.col("pipeline_deleted_at"),
                F.col("databricks_cost"),
                F.col("update_cost"),
                F.col("maintenance_cost"),
                # Cloud EC2/EBS attributed in step 1b above. NULL for
                # serverless (no separate VM line) and for classic pipeline-
                # days whose explorer rows have not landed yet (plan §5); the
                # total_cost COALESCE keeps the total safe in both cases.
                F.col("cloud_cost"),
                F.col("currency"),
                F.col("sku_name"),
                F.col("billing_origin_product"),
            ]

            final_df = joined.select(*select_cols)

            final_df = (
                final_df
                .withColumn(
                    "total_cost",
                    F.coalesce(F.col("databricks_cost"), F.lit(0.0))
                    + F.coalesce(F.col("cloud_cost"), F.lit(0.0)),
                )
                .withColumn("created_at", F.current_timestamp())
                .withColumn("updated_at", F.current_timestamp())
            )
            final_df = safe_cache(final_df)

            row_count = final_df.count()

            validate_source_schema(
                final_df,
                {"workspace_id": "string", "pipeline_id": "string",
                 "usage_date": "date", "billing_origin_product": "string",
                 "workload_type": "string", "compute_mode": "string",
                 "cost_basis": "string", "metadata_missing": "boolean",
                 "databricks_cost": "double", "cloud_cost": "double",
                 "total_cost": "double"},
                self.target_table, self.logger,
            )
            validate_no_negative_costs(
                final_df,
                ["databricks_cost", "update_cost", "maintenance_cost",
                 "cloud_cost", "total_cost"],
                self.target_table, self.logger,
            )
            validate_currency_consistency(final_df, "currency", self.target_table, self.logger)

            target = DeltaTable.forName(spark, self.target_table)
            (target.alias("t")
                .merge(
                    final_df.alias("s"),
                    # All four key columns are NON-nullable, so plain '=' is
                    # safe (the null-safe cluster_id concern is staging-only,
                    # §5.4 - cluster_id is aggregated out of the rollup key).
                    "t.workspace_id = s.workspace_id "
                    "AND t.pipeline_id = s.pipeline_id "
                    "AND t.usage_date = s.usage_date "
                    "AND t.billing_origin_product = s.billing_origin_product",
                )
                .whenMatchedUpdate(set={
                    "pipeline_name": "s.pipeline_name",
                    "pipeline_type": "s.pipeline_type",
                    "created_by": "s.created_by",
                    "run_as": "s.run_as",
                    "workload_type": "s.workload_type",
                    "compute_mode": "s.compute_mode",
                    "cost_basis": "s.cost_basis",
                    "metadata_missing": "s.metadata_missing",
                    "pipeline_deleted_at": "s.pipeline_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "update_cost": "s.update_cost",
                    "maintenance_cost": "s.maintenance_cost",
                    "cloud_cost": "s.cloud_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "updated_at": "current_timestamp()",
                })
                .whenNotMatchedInsert(values={
                    "workspace_id": "s.workspace_id",
                    "pipeline_id": "s.pipeline_id",
                    "usage_date": "s.usage_date",
                    "pipeline_name": "s.pipeline_name",
                    "pipeline_type": "s.pipeline_type",
                    "created_by": "s.created_by",
                    "run_as": "s.run_as",
                    "workload_type": "s.workload_type",
                    "compute_mode": "s.compute_mode",
                    "cost_basis": "s.cost_basis",
                    "metadata_missing": "s.metadata_missing",
                    "pipeline_deleted_at": "s.pipeline_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "update_cost": "s.update_cost",
                    "maintenance_cost": "s.maintenance_cost",
                    "cloud_cost": "s.cloud_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "billing_origin_product": "s.billing_origin_product",
                    "created_at": "current_timestamp()",
                    "updated_at": "current_timestamp()",
                })
                .execute()
            )

            safe_unpersist(final_df)
            safe_unpersist(attributed)
            get_merge_metrics(self.target_table, self.logger)

            validate_post_merge(
                self.target_table, "usage_date",
                start_dt, end_dt, row_count, self.logger,
            )

            # Reconciliation already asserted on the intermediate attribution
            # (§3.3) before the MERGE; the rollup drops cluster_id so it cannot
            # be re-checked on the final table.
            log_audit_run(
                self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                "SUCCESS", row_count, "",
            )
            self.logger.info(
                f"Merged {row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class TotalPipelineSpendsApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        ov_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        self.client = TotalPipelineSpendsClient(
            audit_table=build_table_fqn(catalog, schema, "dbspend360_audit_log"),
            cloud_cost_table=build_table_fqn(catalog, schema, "dbspend360_cloud_cost_explorer"),
            databricks_cost_table=build_table_fqn(catalog, schema, "dbspend360_pipeline_dbu_cost"),
            target_table=build_table_fqn(catalog, schema, "dbspend360_total_pipeline_spends"),
            error_log_table=build_table_fqn(catalog, schema, "dbspend360_error_log"),
            overlap_days=ov_days,
            logger=logger,
        )

    def run(self):
        self.client.build_total_pipeline_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = TotalPipelineSpendsApp()
app.run()